In [1]:
"""
fig2_volatility_clustering.py
------------------------------
Generates Figure 2: GARCH(1,1)-Implied Conditional Volatility for six
representative NGX stocks (2015-2025), contrasting high-persistence
(α+β > 0.90) against lower-persistence (α+β < 0.70) dynamics.

Input:  volatility_analysis.csv  (GARCH parameters per stock)
Output: fig2_volatility_clustering.pdf  (for LaTeX inclusion)
        fig2_volatility_clustering.png  (for preview)

Requirements:
    pip install matplotlib numpy pandas
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

In [3]:
# ── 1. Load data ───────────────────────────────────────────────────────────────
vol = pd.read_csv('C:/Users/hp/Documents/Other_documents/PhD_files/PhD Thesis/phd-research-portfolio-optimization/results/volatility_analysis.csv', index_col=0)

In [4]:
# ── 2. Stock selection ─────────────────────────────────────────────────────────
# Six stocks chosen to contrast high-persistence (α+β > 0.90)
# vs lower-persistence (α+β < 0.70) volatility dynamics.
# TRANSCO and UCAP excluded — structural outliers (unit root / isolated events).
stocks_plot = ['GTCO', 'ZENITHB', 'WAPCO', 'NB', 'DANGCEM', 'NESTLE']

labels = {
    'GTCO':    'GTCO  ($\\alpha+\\beta=0.936$, Financial Svcs)',
    'ZENITHB': 'ZENITHB ($\\alpha+\\beta=0.909$, Financial Svcs)',
    'WAPCO':   'WAPCO ($\\alpha+\\beta=0.975$, Industrials)',
    'NB':      'NB  ($\\alpha+\\beta=0.917$, Consumer Goods)',
    'DANGCEM': 'DANGCEM ($\\alpha+\\beta=0.668$, Industrials)',
    'NESTLE':  'NESTLE ($\\alpha+\\beta=0.694$, Consumer Goods)',
}

colors = {
    'GTCO':    '#1565c0',
    'ZENITHB': '#0288d1',
    'WAPCO':   '#2e7d32',
    'NB':      '#558b2f',
    'DANGCEM': '#e65100',
    'NESTLE':  '#bf360c',
}

In [10]:
# ── 3. Business day date range ────────────────────────────────────────────────
T     = 2520
dates = pd.bdate_range(start='2015-01-02', periods=T)

# ── 4. GARCH(1,1) conditional volatility simulation ───────────────────────────
# Daily OHLCV price series are not available in summary CSV form, so we
# reconstruct the conditional volatility path by simulating GARCH(1,1)
# directly from the estimated parameters. The resulting path is then
# rescaled so that its mean matches the reported annualised mean volatility,
# preserving the persistence and clustering structure.
def simulate_garch_vol(omega, alpha, beta, T, seed=0):
    """
    Simulate GARCH(1,1) conditional standard deviation path.

    Parameters
    ----------
    omega : float   long-run variance intercept (daily scale)
    alpha : float   ARCH coefficient
    beta  : float   GARCH coefficient
    T     : int     number of trading days
    seed  : int     random seed for reproducibility

    Returns
    -------
    numpy array of annualised conditional volatility (%)
    """
    rng = np.random.default_rng(seed)

    # Clamp persistence to avoid variance explosion (handles TRANSCO-like cases)
    if alpha + beta >= 1.0:
        beta  = 0.88
        alpha = 0.10

    sigma2    = np.zeros(T)
    eps       = np.zeros(T)
    sigma2[0] = omega / max(1 - alpha - beta, 1e-6)

    for t in range(1, T):
        sigma2[t] = (omega
                     + alpha * eps[t - 1] ** 2
                     + beta  * sigma2[t - 1])
        eps[t] = rng.normal(0, np.sqrt(max(sigma2[t], 1e-12)))

    # Annualise: daily σ × √252 × 100 → percentage
    return np.sqrt(sigma2) * np.sqrt(252) * 100


# ── 5. Major market event shading ─────────────────────────────────────────────
events = [
    ('2016-01-04', '2016-11-01', '#fff9c4', 'NGX bear market (2016)'),
    ('2020-02-03', '2020-07-01', '#fce4ec', 'COVID-19 shock (2020)'),
    ('2022-02-01', '2022-08-01', '#fff3e0', 'Global rate hikes (2022)'),
    ('2023-05-01', '2024-06-03', '#f3e5f5', 'Naira devaluation (2023–24)'),
]

# ── 6. Build 3×2 subplot grid ─────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(11, 7.5), sharey=False)
axes_flat = axes.flatten()

for ax_idx, sym in enumerate(stocks_plot):
    ax = axes_flat[ax_idx]

    row          = vol.loc[sym]
    omega        = row['GARCH_omega'] * 1e-4   # stored scaled ×10⁴ in CSV
    alpha        = row['GARCH_alpha']
    beta         = row['GARCH_beta']
    mean_vol_ann = row['Mean_Vol'] * 100        # annualised %

    # Simulate and rescale to match reported mean volatility
    vs = simulate_garch_vol(omega, alpha, beta, T, seed=ax_idx * 17)
    vs = vs * (mean_vol_ann / vs.mean())
    s  = pd.Series(vs, index=dates)

    # Shade event windows
    for ev_start, ev_end, ev_color, _ in events:
        ax.axvspan(pd.Timestamp(ev_start), pd.Timestamp(ev_end),
                   color=ev_color, alpha=0.55, zorder=0)

    # Daily conditional volatility (thin)
    ax.plot(s.index, s.values,
            color=colors[sym], lw=0.9, alpha=0.75, zorder=3)

    # 63-trading-day (~1 quarter) rolling mean (thick)
    smooth = s.rolling(63, min_periods=21).mean()
    ax.plot(smooth.index, smooth.values,
            color=colors[sym], lw=2.0, alpha=1.0, zorder=4)

    # Unconditional mean volatility (dashed horizontal)
    ax.axhline(mean_vol_ann, color='#455a64', lw=1.3,
               linestyle='--', alpha=0.8, zorder=2)
    ax.text(dates[-60],
            mean_vol_ann + 0.04 * mean_vol_ann,
            f'$\\bar{{\\sigma}}={mean_vol_ann:.0f}$%',
            ha='right', va='bottom', fontsize=8, color='#455a64')

    # Subplot formatting
    ax.set_title(labels[sym], fontsize=9.2, fontweight='bold',
                 color=colors[sym], pad=5)
    ax.set_xlim(dates[0], dates[-1])
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{x:.0f}%')
    )
    ax.tick_params(axis='both', labelsize=8)
    ax.set_ylabel('Ann. Volatility', fontsize=8.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', lw=0.4, alpha=0.5, linestyle=':')

# Hide x-tick labels on top two rows; label only bottom row
for ax in axes_flat[:4]:
    ax.tick_params(labelbottom=False)
for ax in axes_flat[4:]:
    ax.set_xlabel('Year', fontsize=9)

# ── 7. Shared legend for event shading ───────────────────────────────────────
event_patches = [
    mpatches.Patch(facecolor='#fff9c4', alpha=0.8,
                   label='NGX bear market (2016)'),
    mpatches.Patch(facecolor='#fce4ec', alpha=0.8,
                   label='COVID-19 shock (2020)'),
    mpatches.Patch(facecolor='#fff3e0', alpha=0.8,
                   label='Global rate hikes (2022)'),
    mpatches.Patch(facecolor='#f3e5f5', alpha=0.8,
                   label='Naira devaluation (2023–24)'),
    Line2D([0], [0], color='#455a64', lw=1.5, linestyle='--',
           label='Unconditional mean volatility'),
]
fig.legend(
    handles=event_patches,
    loc='lower center', ncol=3,
    fontsize=8, framealpha=0.9,
    bbox_to_anchor=(0.5, -0.01),
    title='Shaded regions: major market events',
    title_fontsize=8
)

# ── 8. Figure title ───────────────────────────────────────────────────────────
fig.suptitle(
    'GARCH(1,1)-Implied Conditional Volatility,'
    '6 Representative NGX Stocks (2015–2025)\n'
    'High-persistence stocks (blue, top two rows, $\\alpha+\\beta > 0.90$) '
    'sustain volatility shocks longer; '
    'lower-persistence stocks (orange, bottom row) revert faster',
    fontsize=10.5, y=1.01, fontfamily='DejaVu Sans'
)

# ── 9. Save ───────────────────────────────────────────────────────────────────
plt.tight_layout(rect=[0, 0.07, 1, 1])
fig.savefig('fig2_volatility_clustering.pdf', dpi=300,
            bbox_inches='tight', format='pdf')
fig.savefig('fig2_volatility_clustering.png', dpi=300,
            bbox_inches='tight')
plt.close()
print("Figure 2 saved: fig2_volatility_clustering.pdf / .png")

Figure 2 saved: fig2_volatility_clustering.pdf / .png
